## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [7]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [8]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [9]:
# checking the distribution on each of category in categorical variable
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

## Feature Engineering - 1st Digit and 2 first Digit - Benford's Law

In [10]:
def extract_first_digit_features(df, numeric_cols,n=1):
    """Extracts the leading digits (e.g., 1st digit or first 2 digits)

    from numeric columns for Benford's Law analysis.

    Parameters:
    - df: pandas DataFrame
    - numeric_cols: list of columns to process
    - n_digits: number of leading digits to extract (default is 1)
    """
    df = df.copy()

    for col in numeric_cols:
        # 1. Convert to string, strip all non-digit characters (decimals, signs)
        str_series = (
            df[col].astype(str).str.replace(r"\D+", "", regex=True).str.lstrip("0")
        )

        # 2. Extract the first n digits (handling empty strings where values were 0)
        feature_name = (
            f"{col}_first_digit"
        )

        # 3. Extract the first n characters, handle blanks as 'Missing' or NaN,
        # and convert the result into a categorical type
        extracted = (
            str_series.str[:n].replace("", np.nan).fillna("Missing")
        )  # Using string 'Missing' to keep categories clean

        # Extract slice and handle cases where string length is less than n_digits
        df[feature_name] = extracted.astype("category")

    return df

In [11]:
def extract_first_two_digits(df, numeric_cols):
  """Extracts the first two leading digits from numeric columns.

  Handles negatives, decimals, and leading zeros (e.g., 0.0034 -> 34).

  Parameters:
  - df: pandas DataFrame
  - numeric_cols: list of column names to process

  Returns:
  - DataFrame with new columns named '{col}_first_two_digits'
  """
  df = df.copy()

  for col in numeric_cols:
    # 1. Convert to string and remove non-digit characters (signs, decimals, exponents)
    # Then strip any leading zeros
    str_series = (
        df[col].astype(str).str.replace(r"[^\d]", "", regex=True).str.lstrip("0")
    )

    # 2. Extract the first 2 characters, handling empty strings (e.g., original value was 0)
    extracted = str_series.str[:2]

    # 3. Format into a clean numeric feature (using -1 for missing/zero values)
    feature_name = f"{col}_first_two_digits"
    df[feature_name] = (
        extracted.replace("", np.nan).fillna("Missing").astype("category")
    )

  return df

In [12]:
numeric_features = ["amount_usd", "account_balance_usd"]

In [13]:
train_featured = extract_first_digit_features(train, numeric_features)

In [14]:
train_featured = extract_first_two_digits(train_featured, numeric_features)

In [15]:
test_featured = extract_first_digit_features(test, numeric_features)

In [16]:
test_featured = extract_first_two_digits(test_featured, numeric_features)

In [19]:
cat_cols = train_featured.select_dtypes(include='category').columns.tolist()

In [20]:
cat_cols

['amount_usd_first_digit',
 'account_balance_usd_first_digit',
 'amount_usd_first_two_digits',
 'account_balance_usd_first_two_digits']

In [21]:
category_cols = obj_cols + cat_cols

In [22]:
category_cols

['merchant_category',
 'card_type',
 'auth_method',
 'channel',
 'device_type',
 'amount_usd_first_digit',
 'account_balance_usd_first_digit',
 'amount_usd_first_two_digits',
 'account_balance_usd_first_two_digits']

In [23]:
# cross tab cat_cols with output
def analyze_categorical_fraud(df, cat_cols, target_col="is_fraud"):
  """Generates cross-tabulations (counts and percentages) between

  categorical columns and the target (is_fraud) column.

  Parameters:
  - df: pandas DataFrame
  - cat_cols: list of categorical column names
  - target_col: name of the target column (default 'is_fraud')
  """
  fraud_analysis = {}

  for col in cat_cols:
    print(f"========================================")
    print(f"Cross-Tabulation for: {col} vs {target_col}")
    print(f"========================================")

    # 1. Absolute counts cross-tabulation (with margins/totals)
    count_tab = pd.crosstab(df[col], df[target_col], margins=True)

    # 2. Row percentages (shows % of fraud vs non-fraud within each category)
    pct_tab = pd.crosstab(
        df[col], df[target_col], normalize="index", margins=True
    ) * 100

    # Combine or print them clearly
    print("\n--- Counts ---")
    print(count_tab)

    print("\n--- Row Percentages (%) ---")
    print(pct_tab.round(2))
    print("\n")

    # Store in a dictionary if you want to inspect them later programmatically
    fraud_analysis[col] = {"counts": count_tab, "percentages": pct_tab}

  return fraud_analysis

In [24]:
fr_analysis = analyze_categorical_fraud(train_featured, category_cols, target_col="is_fraud")

Cross-Tabulation for: merchant_category vs is_fraud

--- Counts ---
is_fraud              0.0    1.0     All
merchant_category                       
Crypto Exchange      4940    345    5285
Electronics         47192   1501   48693
Fuel                59139    741   59880
Gaming              22473   1311   23784
Gift Cards           8766    690    9456
Groceries          140993   1585  142578
Healthcare          35427    462   35889
Online Retail      153378   1719  155097
Restaurants        100201   1159  101360
Streaming           46135   1061   47196
Travel              28181    534   28715
Utilities           41446    581   42027
All                688271  11689  699960

--- Row Percentages (%) ---
is_fraud             0.0   1.0
merchant_category             
Crypto Exchange    93.47  6.53
Electronics        96.92  3.08
Fuel               98.76  1.24
Gaming             94.49  5.51
Gift Cards         92.70  7.30
Groceries          98.89  1.11
Healthcare         98.71  1.29
Online Re

In [25]:
# fr_analysis

## Model using XGboost + TE + FE Digit

In [26]:
from sklearn.preprocessing import TargetEncoder

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [29]:
import xgboost as xgb

In [30]:
X = train_featured.drop(columns=["id","is_fraud"]).copy()
y = train_featured["is_fraud"].copy()

### Insert TE in the run_cv loop

In [2]:
best_params = {'learning_rate': 0.023621412607945612, 'max_depth': 8, 'reg_lambda': 0.15767822496360231, 'reg_alpha': 0.3574854835852679, 'min_child_weight': 29}

In [4]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        **best_params
}

In [5]:
final_params

{'objective': 'binary:logistic',
 'eval_metric': 'auc',
 'n_estimators': 5000,
 'tree_method': 'hist',
 'enable_categorical': True,
 'random_state': 42,
 'colsample_bytree': 0.8,
 'subsample': 0.8,
 'n_jobs': -1,
 'early_stopping_rounds': 200,
 'learning_rate': 0.023621412607945612,
 'max_depth': 8,
 'reg_lambda': 0.15767822496360231,
 'alpha': 0.3574854835852679,
 'min_child_weight': 29}

In [41]:
def run_cv(fit_predict_fn, te_cols, name): # adding input te_cols for categorical variable required for TE
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        # --- FIX: Fill NaNs and ensure string type for TE columns ---
        #for col in te_cols:
        #    X_tr.loc[:,col] = X_tr.loc[:,col].fillna("Missing").astype(str)
        #   X_va.loc[:,col] = X_va.loc[:,col].fillna("Missing").astype(str)
        #   X_test.loc[:,col] = X_test.loc[:,col].fillna("Missing").astype(str)
        
        # -----------------------------------------------------------
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])
        
        # transform on X_test data
        X_test = test_featured.drop(columns=["id"]).copy()
        # X_test_obj = X_test[obj_cols].copy()
        # print(X_test.info())
        X_test.loc[:, te_cols] = te.transform(X_test[te_cols])

        # print(f'x_test',X_test.head(2))

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
            # X_tr.loc[:,col] = X_tr.loc[:,col].astype('float64')
            # X_va.loc[:,col] = X_va.loc[:,col].astype('float64')
            # X_test.loc[:,col] = X_test.loc[:,col].astype('float64')

        # print(f'x_tr ', X_tr.info())
        # print(f'x_va ', X_va.info())
        # print(f'x_test ', X_test.info())
        # start predit
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)

        # save prediction in oof and in test
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [42]:
def fit_xgb(X_tr, y_tr, X_va, y_va,X_test):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [47]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(competition_path/name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_xgb
    oof_df.to_csv(competition_path/name_2,index=False)
    print('save:', name_2)

In [44]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [45]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, category_cols, 'XGBoost')

C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01705231 0.01489253 0.01705231 ... 0.01361512 0.01691813 0.01361512]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01686482 0.01721842 0.01686482 ... 0.01572138 0.01686482 0.01572138]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01710454 0

  fold 0: AUC = 0.94966


C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01715972 0.01353566 0.0169244  ... 0.01473854 0.0169244  0.01353566]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01671876 0.01554197 0.01671876 ... 0.01651963 0.01671876 0.01554197]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01671422 0

  fold 1: AUC = 0.95126


C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01495316 0.01791196 0.01316959 ... 0.01495316 0.01689685 0.01316959]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01778266 0.01663112 0.01526737 ... 0.01750058 0.01663112 0.01526737]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0140325  0

  fold 2: AUC = 0.94446


C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01663914 0.01497227 0.01663914 ... 0.01497227 0.01691696 0.01351596]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01692513 0.01771679 0.01692513 ... 0.01646204 0.01692513 0.01583317]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01815184 0

  fold 3: AUC = 0.94428


C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01845413 0.01510457 0.01845413 ... 0.01911335 0.01362294 0.01510457]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01681122 0.01722763 0.01681122 ... 0.01681122 0.01543613 0.01583212]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_11276\3805973123.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01928366 0

  fold 4: AUC = 0.94271
XGBoost | OOF AUC = 0.94633 (mean 0.94647 ± 0.00335)


In [49]:
save_sub(test_xgb, oof_xgb, f'sub_xgb_te_digitv3_{SEED}.csv', f'oof_xgb_te_digitv3_{SEED}.csv')

save: sub_xgb_te_digitv3_2312.csv
save: oof_xgb_te_digitv3_2312.csv


In [25]:
seed = [42,67,1508,2312,106]

In [26]:
auc_xgb_dict = {}

In [28]:
for Seed in seed:
    SEED = Seed
    N_FOLDS = 5
    TARGET = 'is_fraud'

    # Stratified KFold cross validation
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    FOLDS = list(skf.split(X, y))

    oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, obj_cols, 'XGBoost')
    save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_{SEED}.csv', f'oof/oof_xgb_te_{SEED}.csv')
    auc_xgb_dict[Seed] = auc_xgb

  fold 0: AUC = 0.94455
  fold 1: AUC = 0.94846
  fold 2: AUC = 0.95111
  fold 3: AUC = 0.94614
  fold 4: AUC = 0.94221
XGBoost | OOF AUC = 0.94646 (mean 0.94649 ± 0.00308)
save: sub/sub_xgb_te_42.csv
save: oof/oof_xgb_te_42.csv
  fold 0: AUC = 0.94719
  fold 1: AUC = 0.94695
  fold 2: AUC = 0.94497
  fold 3: AUC = 0.94703
  fold 4: AUC = 0.94649
XGBoost | OOF AUC = 0.94650 (mean 0.94653 ± 0.00081)
save: sub/sub_xgb_te_67.csv
save: oof/oof_xgb_te_67.csv
  fold 0: AUC = 0.94478
  fold 1: AUC = 0.94771
  fold 2: AUC = 0.94872
  fold 3: AUC = 0.94711
  fold 4: AUC = 0.94394
XGBoost | OOF AUC = 0.94642 (mean 0.94645 ± 0.00180)
save: sub/sub_xgb_te_1508.csv
save: oof/oof_xgb_te_1508.csv
  fold 0: AUC = 0.95012
  fold 1: AUC = 0.95131
  fold 2: AUC = 0.94461
  fold 3: AUC = 0.94382
  fold 4: AUC = 0.94205
XGBoost | OOF AUC = 0.94630 (mean 0.94638 ± 0.00365)
save: sub/sub_xgb_te_2312.csv
save: oof/oof_xgb_te_2312.csv
  fold 0: AUC = 0.94696
  fold 1: AUC = 0.94823
  fold 2: AUC = 0.94669
  fo